# MB25 — Centro 4014 — Datalake

**Tabela:** `prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material`
**Ambiente:** 🔴 **PRODUÇÃO (PRD)** · **Objeto:** VIEW
**Filtro do cenário:** `cod_centro = '4014'`
**Volume medido na exploratória:** 5.879 linhas · 1.953 reservas · 2,94% da view

---

## Como usar

Aperte **Run All**. Células **independentes** e **somente leitura** — nenhum `CREATE`, `INSERT`
ou gravação. Sem widget, sem view temporária, sem ordem obrigatória.

## ⚠️ Leia antes de extrair o SAP — a view NÃO é a MB25

O comentário da própria view declara que ela é a **base RESB sem os filtros da tela de seleção**.
Para reproduzir o resultado da MB25 é preciso aplicar:

| Filtro MB25 | Campo SAP | Na view |
|---|---|---|
| Movimento de mercadoria permitido | XWAOK | `ind_movimento_permitido = 'X'` |
| Somente reservas em aberto (P_OPEN) | — | `qt_saldo_reserva > 0` |
| Intervalo de data de necessidade | BDTER | `dt_necessidade` |
| Centro | WERKS | `cod_centro` |

Na exploratória, **só 17% das linhas têm `ind_movimento_permitido = 'X'`**. Comparar a view inteira
com a MB25 vai acusar ~80% de "sobra" que não é erro. Por isso este notebook entrega **duas extrações**:

- **19.1 — base completa do centro** (para comparar com RESB/SE16)
- **19.2 — equivalente MB25** (para comparar com a tela MB25 com os filtros acima)

## Achados da exploratória que afetam este cenário

| Severidade | Achado |
|---|---|
| 🔴 ALTA | **Chave não é única.** `reserva + item` tem 704 linhas a mais na view toda: **521 são 100% idênticas** (duplicata pura) e ~183 são **versões diferentes** do mesmo item (mudam material, datas e quantidades, mesma `dateingest`). |
| 🔴 ALTA | **~32% das linhas sem cabeçalho.** 63.740 linhas vêm com `dt_reserva`, `num_ordem`, `cod_centro_custo`, `nm_usuario` e demais campos de cabeçalho **nulos** — o join com o cabeçalho da reserva (RKPF) não encontrou par. |
| 🟡 MÉDIA | **`tp_registro` (RSART) 100% vazio** — mas faz parte da chave declarada. |
| 🟡 MÉDIA | **`ind_item_eliminado` e `ind_registro_final` 100% vazios** — coerente: a view já filtra eliminados e finalizados. |
| 🟡 MÉDIA | **2.898 linhas com movimento NÃO permitido mas com retirada.** |
| 🟡 MÉDIA | **`dt_reserva = '00000000'` em 125.640 linhas** + 63.740 nulas — só ~5% têm data útil. |
| ⚪ BAIXA | **`qt_saldo_reserva` confere 100%** com `qt_necessaria − qt_retirada`. Replicação de cabeçalho nos itens confirmada. |

## Cuidados de PRD

- Dados pessoais (`nm_usuario`, `nm_recebedor_mercadoria`) **mascarados por hash** em amostras e extrações.
- Várias células leem o recorte inteiro — rode fora do pico se o cluster for compartilhado.


## 1. Metadados

Objeto é **view** — `DESCRIBE DETAIL`/`HISTORY` comentados para não quebrar o Run All.

In [ ]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

In [ ]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- DESCRIBE DETAIL prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;
-- DESCRIBE HISTORY prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria e representatividade

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)                                   AS linhas_view_toda,
  (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')                               AS linhas_centro_4014,
  ROUND(100.0 * (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014') / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct_do_total,
  (SELECT COUNT(DISTINCT num_reserva) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')            AS reservas,
  (SELECT COUNT(DISTINCT cod_material) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')           AS materiais,
  (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0)                              AS linhas_equivalente_mb25;

## 3. Confirmação do filtro

**Se retornar mais de uma linha com formatos diferentes, o filtro `= '4014'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(cod_centro AS STRING) AS valor_bruto, length(cod_centro) AS comprimento, COUNT(*) AS linhas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
WHERE regexp_replace(trim(cod_centro), '^0+', '') = '4014' OR trim(cod_centro) = '4014'
GROUP BY cod_centro, length(cod_centro)
ORDER BY linhas DESC;

## 3.1 🔴 Funil: view → equivalente MB25

Mostra quantas linhas sobram a cada filtro da tela MB25. **O número final é o que a MB25 deve
mostrar** para este centro (sem filtro de data de necessidade).

In [ ]:
-- 3.1 FUNIL VIEW -> MB25
SELECT '1. view - centro 4014' AS etapa, COUNT(*) AS linhas, COUNT(DISTINCT num_reserva) AS reservas
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
UNION ALL
SELECT '2. + movimento permitido (XWAOK = X)', COUNT(*), COUNT(DISTINCT num_reserva)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X'
UNION ALL
SELECT '3. + saldo em aberto (qt_saldo_reserva > 0)', COUNT(*), COUNT(DISTINCT num_reserva)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0
UNION ALL
SELECT '(ref) somente saldo > 0, sem XWAOK', COUNT(*), COUNT(DISTINCT num_reserva)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND qt_saldo_reserva > 0
ORDER BY etapa;

In [ ]:
-- 3.2 FAIXA DE dt_necessidade NO EQUIVALENTE MB25
-- Use para definir o intervalo BDTER na tela da MB25.
SELECT MIN(dt_necessidade) AS necessidade_min,
       MAX(dt_necessidade) AS necessidade_max,
       COUNT(DISTINCT dt_necessidade) AS datas_distintas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0;

## 4. Granularidade e chave real

Na view toda, **nenhuma chave foi única** (1,0035 linhas/chave). Testamos também material e
`dt_criacao`, que variaram entre as duplicatas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva + tp_registro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `tp_registro` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva + cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva + cod_material + dt_criacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material`, `dt_criacao` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
)
SELECT g.chave, t.total AS linhas, g.distintos, ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA' ELSE 'NAO UNICA' END AS veredito
FROM g CROSS JOIN t ORDER BY linhas_por_chave;

## 5. 🔴 Duplicidade — pura × versão

Separa os dois tipos encontrados na exploratória:

- **Duplicata pura:** linha 100% idêntica repetida → erro de carga, sem discussão.
- **Versão:** mesma `reserva + item` com valores diferentes → o pipeline manteve a versão antiga
  junto com a nova. **No SAP só existe uma** — a comparação vai acusar sobra.

In [ ]:
-- 5. DUPLICIDADE: PURA x VERSAO
WITH h AS (
  SELECT num_reserva, num_item_reserva, md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))) AS hash_linha
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
),
k AS (
  SELECT num_reserva, num_item_reserva, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes
  FROM h GROUP BY num_reserva, num_item_reserva
)
SELECT COUNT(*)                                        AS chaves,
       SUM(linhas)                                     AS linhas,
       COUNT_IF(linhas > 1)                            AS chaves_duplicadas,
       SUM(linhas) - SUM(versoes)                      AS linhas_duplicata_pura,
       COUNT_IF(versoes > 1)                           AS chaves_com_mais_de_uma_versao,
       SUM(CASE WHEN versoes > 1 THEN versoes - 1 ELSE 0 END) AS linhas_versao_excedente
FROM k;

In [ ]:
-- 5.1 EXEMPLOS DE CHAVES DUPLICADAS
SELECT `num_reserva`, `num_item_reserva`, COUNT(*) AS qtd
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY `num_reserva`, `num_item_reserva` HAVING COUNT(*) > 1
ORDER BY qtd DESC, num_reserva LIMIT 30;

In [ ]:
-- 5.2 O QUE MUDA ENTRE AS VERSOES
WITH cen AS (SELECT * FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'),
dup AS (SELECT `num_reserva`, `num_item_reserva` FROM cen GROUP BY `num_reserva`, `num_item_reserva` HAVING COUNT(*) > 1),
d AS (SELECT c.* FROM cen c JOIN dup ON c.num_reserva <=> dup.num_reserva AND c.num_item_reserva <=> dup.num_item_reserva),
agg AS (SELECT `num_reserva`, `num_item_reserva`,
         COUNT(DISTINCT `tp_registro`) AS `tp_registro`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `num_lote`) AS `num_lote`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `cod_centro_receptor`) AS `cod_centro_receptor`,
         COUNT(DISTINCT `cod_deposito_receptor`) AS `cod_deposito_receptor`,
         COUNT(DISTINCT `dt_necessidade`) AS `dt_necessidade`,
         COUNT(DISTINCT `dt_reserva`) AS `dt_reserva`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `qt_necessaria`) AS `qt_necessaria`,
         COUNT(DISTINCT `qt_retirada`) AS `qt_retirada`,
         COUNT(DISTINCT `qt_saldo_reserva`) AS `qt_saldo_reserva`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `qt_um_registro`) AS `qt_um_registro`,
         COUNT(DISTINCT `sg_unidade_medida_registro`) AS `sg_unidade_medida_registro`,
         COUNT(DISTINCT `qt_verificacao_disponibilidade`) AS `qt_verificacao_disponibilidade`,
         COUNT(DISTINCT `num_ordem`) AS `num_ordem`,
         COUNT(DISTINCT `cod_operacao`) AS `cod_operacao`,
         COUNT(DISTINCT `tp_movimento`) AS `tp_movimento`,
         COUNT(DISTINCT `tp_necessidade`) AS `tp_necessidade`,
         COUNT(DISTINCT `cod_debito_credito`) AS `cod_debito_credito`,
         COUNT(DISTINCT `num_ordem_referencia`) AS `num_ordem_referencia`,
         COUNT(DISTINCT `num_ordem_servico`) AS `num_ordem_servico`,
         COUNT(DISTINCT `num_item_ordem_servico`) AS `num_item_ordem_servico`,
         COUNT(DISTINCT `st_reserva`) AS `st_reserva`,
         COUNT(DISTINCT `ind_movimento_permitido`) AS `ind_movimento_permitido`,
         COUNT(DISTINCT `ind_item_eliminado`) AS `ind_item_eliminado`,
         COUNT(DISTINCT `ind_registro_final`) AS `ind_registro_final`,
         COUNT(DISTINCT `ind_item_dummy`) AS `ind_item_dummy`,
         COUNT(DISTINCT `ind_material_granel`) AS `ind_material_granel`,
         COUNT(DISTINCT `cod_estoque_especial`) AS `cod_estoque_especial`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `num_imobilizado`) AS `num_imobilizado`,
         COUNT(DISTINCT `num_sub_imobilizado`) AS `num_sub_imobilizado`,
         COUNT(DISTINCT `num_diagrama_rede`) AS `num_diagrama_rede`,
         COUNT(DISTINCT `num_ordem_cliente`) AS `num_ordem_cliente`,
         COUNT(DISTINCT `num_item_ordem_cliente`) AS `num_item_ordem_cliente`,
         COUNT(DISTINCT `num_divisao_ordem_cliente`) AS `num_divisao_ordem_cliente`,
         COUNT(DISTINCT `desc_observacao`) AS `desc_observacao`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `num_reserva`, `num_item_reserva`)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA entre as linhas' ELSE 'constante' END AS veredito
FROM (SELECT stack(45,
    'tp_registro', MAX(`tp_registro`),
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'num_lote', MAX(`num_lote`),
    'cod_centro', MAX(`cod_centro`),
    'cod_deposito', MAX(`cod_deposito`),
    'cod_centro_receptor', MAX(`cod_centro_receptor`),
    'cod_deposito_receptor', MAX(`cod_deposito_receptor`),
    'dt_necessidade', MAX(`dt_necessidade`),
    'dt_reserva', MAX(`dt_reserva`),
    'dt_criacao', MAX(`dt_criacao`),
    'qt_necessaria', MAX(`qt_necessaria`),
    'qt_retirada', MAX(`qt_retirada`),
    'qt_saldo_reserva', MAX(`qt_saldo_reserva`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'qt_um_registro', MAX(`qt_um_registro`),
    'sg_unidade_medida_registro', MAX(`sg_unidade_medida_registro`),
    'qt_verificacao_disponibilidade', MAX(`qt_verificacao_disponibilidade`),
    'num_ordem', MAX(`num_ordem`),
    'cod_operacao', MAX(`cod_operacao`),
    'tp_movimento', MAX(`tp_movimento`),
    'tp_necessidade', MAX(`tp_necessidade`),
    'cod_debito_credito', MAX(`cod_debito_credito`),
    'num_ordem_referencia', MAX(`num_ordem_referencia`),
    'num_ordem_servico', MAX(`num_ordem_servico`),
    'num_item_ordem_servico', MAX(`num_item_ordem_servico`),
    'st_reserva', MAX(`st_reserva`),
    'ind_movimento_permitido', MAX(`ind_movimento_permitido`),
    'ind_item_eliminado', MAX(`ind_item_eliminado`),
    'ind_registro_final', MAX(`ind_registro_final`),
    'ind_item_dummy', MAX(`ind_item_dummy`),
    'ind_material_granel', MAX(`ind_material_granel`),
    'cod_estoque_especial', MAX(`cod_estoque_especial`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'num_imobilizado', MAX(`num_imobilizado`),
    'num_sub_imobilizado', MAX(`num_sub_imobilizado`),
    'num_diagrama_rede', MAX(`num_diagrama_rede`),
    'num_ordem_cliente', MAX(`num_ordem_cliente`),
    'num_item_ordem_cliente', MAX(`num_item_ordem_cliente`),
    'num_divisao_ordem_cliente', MAX(`num_divisao_ordem_cliente`),
    'desc_observacao', MAX(`desc_observacao`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos) FROM agg)
ORDER BY max_valores_distintos DESC, coluna;

## 5.3 🔴 Itens sem cabeçalho

Na view toda, ~32% das linhas vieram com todos os campos de cabeçalho nulos. Aqui medimos no
centro e mostramos o perfil dessas linhas.

In [ ]:
-- 5.3 ITENS SEM CABECALHO
SELECT COUNT(*)                                             AS linhas,
       COUNT_IF(dt_reserva IS NULL)                         AS sem_cabecalho,
       ROUND(100.0 * COUNT_IF(dt_reserva IS NULL) / COUNT(*), 2) AS pct_sem_cabecalho,
       COUNT_IF(dt_reserva IS NULL AND num_ordem IS NULL)   AS sem_cabecalho_e_sem_ordem,
       COUNT(DISTINCT CASE WHEN dt_reserva IS NULL THEN num_reserva END) AS reservas_sem_cabecalho
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014';

In [ ]:
-- 5.3b PERFIL DAS LINHAS SEM CABECALHO
SELECT COALESCE(NULLIF(trim(tp_movimento),''),'(vazio)') AS tp_movimento,
       COALESCE(NULLIF(trim(tp_necessidade),''),'(vazio)') AS tp_necessidade,
       COUNT(*) AS linhas, MIN(dt_necessidade) AS necessidade_min, MAX(dt_necessidade) AS necessidade_max
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND dt_reserva IS NULL
GROUP BY 1, 2 ORDER BY linhas DESC LIMIT 20;

## 6. Preenchimento de TODAS as colunas

> Nos indicadores `ind_`, vazio significa **"não"** — o % útil baixo não é erro.

In [ ]:
-- 6. PREENCHIMENTO NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'),
perf AS (SELECT stack(49,
    'num_reserva', 'string', COUNT_IF(`num_reserva` IS NULL), COUNT_IF(`num_reserva` IS NOT NULL AND lower(trim(`num_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_reserva`) RLIKE '^0+([.,]0+)?$'),
    'num_item_reserva', 'string', COUNT_IF(`num_item_reserva` IS NULL), COUNT_IF(`num_item_reserva` IS NOT NULL AND lower(trim(`num_item_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_reserva`) RLIKE '^0+([.,]0+)?$'),
    'tp_registro', 'string', COUNT_IF(`tp_registro` IS NULL), COUNT_IF(`tp_registro` IS NOT NULL AND lower(trim(`tp_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_registro`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'num_lote', 'string', COUNT_IF(`num_lote` IS NULL), COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lote`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_receptor', 'string', COUNT_IF(`cod_centro_receptor` IS NULL), COUNT_IF(`cod_centro_receptor` IS NOT NULL AND lower(trim(`cod_centro_receptor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_receptor`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_receptor', 'string', COUNT_IF(`cod_deposito_receptor` IS NULL), COUNT_IF(`cod_deposito_receptor` IS NOT NULL AND lower(trim(`cod_deposito_receptor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_receptor`) RLIKE '^0+([.,]0+)?$'),
    'dt_necessidade', 'string', COUNT_IF(`dt_necessidade` IS NULL), COUNT_IF(`dt_necessidade` IS NOT NULL AND lower(trim(`dt_necessidade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_necessidade`) RLIKE '^0+([.,]0+)?$'),
    'dt_reserva', 'string', COUNT_IF(`dt_reserva` IS NULL), COUNT_IF(`dt_reserva` IS NOT NULL AND lower(trim(`dt_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_reserva`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'qt_necessaria', 'decimal(13,3)', COUNT_IF(`qt_necessaria` IS NULL), 0L, COUNT_IF(`qt_necessaria` = 0),
    'qt_retirada', 'decimal(13,3)', COUNT_IF(`qt_retirada` IS NULL), 0L, COUNT_IF(`qt_retirada` = 0),
    'qt_saldo_reserva', 'decimal(14,3)', COUNT_IF(`qt_saldo_reserva` IS NULL), 0L, COUNT_IF(`qt_saldo_reserva` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'qt_um_registro', 'decimal(13,3)', COUNT_IF(`qt_um_registro` IS NULL), 0L, COUNT_IF(`qt_um_registro` = 0),
    'sg_unidade_medida_registro', 'string', COUNT_IF(`sg_unidade_medida_registro` IS NULL), COUNT_IF(`sg_unidade_medida_registro` IS NOT NULL AND lower(trim(`sg_unidade_medida_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_registro`) RLIKE '^0+([.,]0+)?$'),
    'qt_verificacao_disponibilidade', 'decimal(15,3)', COUNT_IF(`qt_verificacao_disponibilidade` IS NULL), 0L, COUNT_IF(`qt_verificacao_disponibilidade` = 0),
    'num_ordem', 'string', COUNT_IF(`num_ordem` IS NULL), COUNT_IF(`num_ordem` IS NOT NULL AND lower(trim(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_operacao', 'string', COUNT_IF(`cod_operacao` IS NULL), COUNT_IF(`cod_operacao` IS NOT NULL AND lower(trim(`cod_operacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_operacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_movimento', 'string', COUNT_IF(`tp_movimento` IS NULL), COUNT_IF(`tp_movimento` IS NOT NULL AND lower(trim(`tp_movimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_movimento`) RLIKE '^0+([.,]0+)?$'),
    'tp_necessidade', 'string', COUNT_IF(`tp_necessidade` IS NULL), COUNT_IF(`tp_necessidade` IS NOT NULL AND lower(trim(`tp_necessidade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_necessidade`) RLIKE '^0+([.,]0+)?$'),
    'cod_debito_credito', 'string', COUNT_IF(`cod_debito_credito` IS NULL), COUNT_IF(`cod_debito_credito` IS NOT NULL AND lower(trim(`cod_debito_credito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_debito_credito`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_referencia', 'string', COUNT_IF(`num_ordem_referencia` IS NULL), COUNT_IF(`num_ordem_referencia` IS NOT NULL AND lower(trim(`num_ordem_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_referencia`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_servico', 'string', COUNT_IF(`num_ordem_servico` IS NULL), COUNT_IF(`num_ordem_servico` IS NOT NULL AND lower(trim(`num_ordem_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_servico`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ordem_servico', 'string', COUNT_IF(`num_item_ordem_servico` IS NULL), COUNT_IF(`num_item_ordem_servico` IS NOT NULL AND lower(trim(`num_item_ordem_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ordem_servico`) RLIKE '^0+([.,]0+)?$'),
    'st_reserva', 'string', COUNT_IF(`st_reserva` IS NULL), COUNT_IF(`st_reserva` IS NOT NULL AND lower(trim(`st_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_reserva`) RLIKE '^0+([.,]0+)?$'),
    'ind_movimento_permitido', 'string', COUNT_IF(`ind_movimento_permitido` IS NULL), COUNT_IF(`ind_movimento_permitido` IS NOT NULL AND lower(trim(`ind_movimento_permitido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_movimento_permitido`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_eliminado', 'string', COUNT_IF(`ind_item_eliminado` IS NULL), COUNT_IF(`ind_item_eliminado` IS NOT NULL AND lower(trim(`ind_item_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_eliminado`) RLIKE '^0+([.,]0+)?$'),
    'ind_registro_final', 'string', COUNT_IF(`ind_registro_final` IS NULL), COUNT_IF(`ind_registro_final` IS NOT NULL AND lower(trim(`ind_registro_final`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_registro_final`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_dummy', 'string', COUNT_IF(`ind_item_dummy` IS NULL), COUNT_IF(`ind_item_dummy` IS NOT NULL AND lower(trim(`ind_item_dummy`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_dummy`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_granel', 'string', COUNT_IF(`ind_material_granel` IS NULL), COUNT_IF(`ind_material_granel` IS NOT NULL AND lower(trim(`ind_material_granel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_material_granel`) RLIKE '^0+([.,]0+)?$'),
    'cod_estoque_especial', 'string', COUNT_IF(`cod_estoque_especial` IS NULL), COUNT_IF(`cod_estoque_especial` IS NOT NULL AND lower(trim(`cod_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'num_imobilizado', 'string', COUNT_IF(`num_imobilizado` IS NULL), COUNT_IF(`num_imobilizado` IS NOT NULL AND lower(trim(`num_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'num_sub_imobilizado', 'string', COUNT_IF(`num_sub_imobilizado` IS NULL), COUNT_IF(`num_sub_imobilizado` IS NOT NULL AND lower(trim(`num_sub_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_sub_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'num_diagrama_rede', 'string', COUNT_IF(`num_diagrama_rede` IS NULL), COUNT_IF(`num_diagrama_rede` IS NOT NULL AND lower(trim(`num_diagrama_rede`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_diagrama_rede`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_cliente', 'string', COUNT_IF(`num_ordem_cliente` IS NULL), COUNT_IF(`num_ordem_cliente` IS NOT NULL AND lower(trim(`num_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ordem_cliente', 'string', COUNT_IF(`num_item_ordem_cliente` IS NULL), COUNT_IF(`num_item_ordem_cliente` IS NOT NULL AND lower(trim(`num_item_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_ordem_cliente', 'string', COUNT_IF(`num_divisao_ordem_cliente` IS NULL), COUNT_IF(`num_divisao_ordem_cliente` IS NOT NULL AND lower(trim(`num_divisao_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'nm_recebedor_mercadoria', 'string', COUNT_IF(`nm_recebedor_mercadoria` IS NULL), COUNT_IF(`nm_recebedor_mercadoria` IS NOT NULL AND lower(trim(`nm_recebedor_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_recebedor_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'desc_observacao', 'string', COUNT_IF(`desc_observacao` IS NULL), COUNT_IF(`desc_observacao` IS NOT NULL AND lower(trim(`desc_observacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_observacao`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0 THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END AS veredito
FROM perf p CROSS JOIN t ORDER BY veredito, pct_util, coluna;

## 8. Domínio das categóricas

In [ ]:
-- 8. DOMINIO
(SELECT 'tp_movimento' AS coluna, CAST(`tp_movimento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `tp_movimento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_necessidade' AS coluna, CAST(`tp_necessidade` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `tp_necessidade` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_debito_credito' AS coluna, CAST(`cod_debito_credito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `cod_debito_credito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_reserva' AS coluna, CAST(`st_reserva` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `st_reserva` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_estoque_especial' AS coluna, CAST(`cod_estoque_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `cod_estoque_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_basica' AS coluna, CAST(`sg_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `sg_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_registro' AS coluna, CAST(`sg_unidade_medida_registro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `sg_unidade_medida_registro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_movimento_permitido' AS coluna, CAST(`ind_movimento_permitido` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `ind_movimento_permitido` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_eliminado' AS coluna, CAST(`ind_item_eliminado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `ind_item_eliminado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_registro_final' AS coluna, CAST(`ind_registro_final` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `ind_registro_final` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_dummy' AS coluna, CAST(`ind_item_dummy` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `ind_item_dummy` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_material_granel' AS coluna, CAST(`ind_material_granel` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY `ind_material_granel` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil numérico

Todos `decimal` — conciliação exata, sem tolerância.

In [ ]:
-- 10. PERFIL NUMERICO
SELECT * FROM (SELECT stack(5,
    'qt_necessaria', COUNT(`qt_necessaria`), CAST(MIN(`qt_necessaria`) AS DOUBLE), CAST(MAX(`qt_necessaria`) AS DOUBLE), CAST(percentile_approx(`qt_necessaria`, 0.5) AS DOUBLE), COUNT_IF(`qt_necessaria` < 0), COUNT_IF(`qt_necessaria` = 0),
    'qt_retirada', COUNT(`qt_retirada`), CAST(MIN(`qt_retirada`) AS DOUBLE), CAST(MAX(`qt_retirada`) AS DOUBLE), CAST(percentile_approx(`qt_retirada`, 0.5) AS DOUBLE), COUNT_IF(`qt_retirada` < 0), COUNT_IF(`qt_retirada` = 0),
    'qt_saldo_reserva', COUNT(`qt_saldo_reserva`), CAST(MIN(`qt_saldo_reserva`) AS DOUBLE), CAST(MAX(`qt_saldo_reserva`) AS DOUBLE), CAST(percentile_approx(`qt_saldo_reserva`, 0.5) AS DOUBLE), COUNT_IF(`qt_saldo_reserva` < 0), COUNT_IF(`qt_saldo_reserva` = 0),
    'qt_um_registro', COUNT(`qt_um_registro`), CAST(MIN(`qt_um_registro`) AS DOUBLE), CAST(MAX(`qt_um_registro`) AS DOUBLE), CAST(percentile_approx(`qt_um_registro`, 0.5) AS DOUBLE), COUNT_IF(`qt_um_registro` < 0), COUNT_IF(`qt_um_registro` = 0),
    'qt_verificacao_disponibilidade', COUNT(`qt_verificacao_disponibilidade`), CAST(MIN(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(MAX(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(percentile_approx(`qt_verificacao_disponibilidade`, 0.5) AS DOUBLE), COUNT_IF(`qt_verificacao_disponibilidade` < 0), COUNT_IF(`qt_verificacao_disponibilidade` = 0)
  ) AS (coluna, preenchidos, minimo, maximo, mediana, negativos, zeros) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

Duas colunas: **base completa** (compare com RESB) e **equivalente MB25** (compare com a tela).
`qt_um_registro` está na unidade de registro; as demais na unidade básica.

In [ ]:
-- 11. TOTAIS: BASE COMPLETA x EQUIVALENTE MB25
WITH a AS (SELECT stack(5,
    'qt_necessaria', CAST(ROUND(SUM(`qt_necessaria`), 3) AS STRING),
    'qt_retirada', CAST(ROUND(SUM(`qt_retirada`), 3) AS STRING),
    'qt_saldo_reserva', CAST(ROUND(SUM(`qt_saldo_reserva`), 3) AS STRING),
    'qt_um_registro', CAST(ROUND(SUM(`qt_um_registro`), 3) AS STRING),
    'qt_verificacao_disponibilidade', CAST(ROUND(SUM(`qt_verificacao_disponibilidade`), 3) AS STRING)
  ) AS (coluna, total_base_completa) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'),
b AS (SELECT stack(5,
    'qt_necessaria', CAST(ROUND(SUM(`qt_necessaria`), 3) AS STRING),
    'qt_retirada', CAST(ROUND(SUM(`qt_retirada`), 3) AS STRING),
    'qt_saldo_reserva', CAST(ROUND(SUM(`qt_saldo_reserva`), 3) AS STRING),
    'qt_um_registro', CAST(ROUND(SUM(`qt_um_registro`), 3) AS STRING),
    'qt_verificacao_disponibilidade', CAST(ROUND(SUM(`qt_verificacao_disponibilidade`), 3) AS STRING)
  ) AS (coluna, total_equivalente_mb25) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0)
SELECT a.coluna, a.total_base_completa, b.total_equivalente_mb25
FROM a JOIN b ON a.coluna = b.coluna ORDER BY a.coluna;

## 12. Datas em STRING

Formato `AAAAMMDD`. **`00000000` = sem data** — no SAP aparece vazio. Normalizar antes de comparar.

In [ ]:
-- 12. DATAS
SELECT * FROM (SELECT stack(3,
    'dt_necessidade', COUNT_IF(`dt_necessidade` IS NULL OR trim(`dt_necessidade`) = ''), COUNT_IF(trim(`dt_necessidade`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_necessidade`) = '00000000'), MIN(CASE WHEN trim(`dt_necessidade`) NOT IN ('', '00000000') THEN `dt_necessidade` END), MAX(CASE WHEN trim(`dt_necessidade`) NOT IN ('', '00000000') THEN `dt_necessidade` END),
    'dt_reserva', COUNT_IF(`dt_reserva` IS NULL OR trim(`dt_reserva`) = ''), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_reserva`) = '00000000'), MIN(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END), MAX(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) = '00000000'), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, data_zero, minimo, maximo) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
ORDER BY coluna;

## 13. Códigos e zeros à esquerda

`cod_material` vem com **18 posições** (`000000000004254340`), `num_reserva` com 10, `num_item_reserva`
com 4 e `num_ordem` com 12. A MB25 exibe sem zeros — normalizar antes do join.

In [ ]:
-- 13. CODIGOS
SELECT coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes
FROM (SELECT stack(10,
    'num_reserva', COUNT_IF(`num_reserva` IS NULL OR trim(`num_reserva`) = ''), MIN(length(trim(`num_reserva`))), MAX(length(trim(`num_reserva`))), COUNT_IF(trim(`num_reserva`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`num_reserva`)), COUNT(DISTINCT regexp_replace(trim(`num_reserva`), '^0+', '')),
    'num_item_reserva', COUNT_IF(`num_item_reserva` IS NULL OR trim(`num_item_reserva`) = ''), MIN(length(trim(`num_item_reserva`))), MAX(length(trim(`num_item_reserva`))), COUNT_IF(trim(`num_item_reserva`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`num_item_reserva`)), COUNT(DISTINCT regexp_replace(trim(`num_item_reserva`), '^0+', '')),
    'cod_material', COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = ''), MIN(length(trim(`cod_material`))), MAX(length(trim(`cod_material`))), COUNT_IF(trim(`cod_material`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_material`)), COUNT(DISTINCT regexp_replace(trim(`cod_material`), '^0+', '')),
    'cod_deposito', COUNT_IF(`cod_deposito` IS NULL OR trim(`cod_deposito`) = ''), MIN(length(trim(`cod_deposito`))), MAX(length(trim(`cod_deposito`))), COUNT_IF(trim(`cod_deposito`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_deposito`)), COUNT(DISTINCT regexp_replace(trim(`cod_deposito`), '^0+', '')),
    'cod_centro_receptor', COUNT_IF(`cod_centro_receptor` IS NULL OR trim(`cod_centro_receptor`) = ''), MIN(length(trim(`cod_centro_receptor`))), MAX(length(trim(`cod_centro_receptor`))), COUNT_IF(trim(`cod_centro_receptor`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_centro_receptor`)), COUNT(DISTINCT regexp_replace(trim(`cod_centro_receptor`), '^0+', '')),
    'num_ordem', COUNT_IF(`num_ordem` IS NULL OR trim(`num_ordem`) = ''), MIN(length(trim(`num_ordem`))), MAX(length(trim(`num_ordem`))), COUNT_IF(trim(`num_ordem`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`num_ordem`)), COUNT(DISTINCT regexp_replace(trim(`num_ordem`), '^0+', '')),
    'num_ordem_referencia', COUNT_IF(`num_ordem_referencia` IS NULL OR trim(`num_ordem_referencia`) = ''), MIN(length(trim(`num_ordem_referencia`))), MAX(length(trim(`num_ordem_referencia`))), COUNT_IF(trim(`num_ordem_referencia`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`num_ordem_referencia`)), COUNT(DISTINCT regexp_replace(trim(`num_ordem_referencia`), '^0+', '')),
    'cod_centro_custo', COUNT_IF(`cod_centro_custo` IS NULL OR trim(`cod_centro_custo`) = ''), MIN(length(trim(`cod_centro_custo`))), MAX(length(trim(`cod_centro_custo`))), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_centro_custo`)), COUNT(DISTINCT regexp_replace(trim(`cod_centro_custo`), '^0+', '')),
    'cod_elemento_pep', COUNT_IF(`cod_elemento_pep` IS NULL OR trim(`cod_elemento_pep`) = ''), MIN(length(trim(`cod_elemento_pep`))), MAX(length(trim(`cod_elemento_pep`))), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_elemento_pep`)), COUNT(DISTINCT regexp_replace(trim(`cod_elemento_pep`), '^0+', '')),
    'num_lote', COUNT_IF(`num_lote` IS NULL OR trim(`num_lote`) = ''), MIN(length(trim(`num_lote`))), MAX(length(trim(`num_lote`))), COUNT_IF(trim(`num_lote`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`num_lote`)), COUNT(DISTINCT regexp_replace(trim(`num_lote`), '^0+', ''))
  ) AS (coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')
ORDER BY coluna;

## 14. Chaves normalizadas para join

Lista pronta para PROCV contra o extrato SAP. A coluna `no_equivalente_mb25` indica se o item
deveria aparecer na tela MB25.

In [ ]:
-- 14. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(num_reserva), '^0+', '')      AS num_reserva_norm,
       regexp_replace(trim(num_item_reserva), '^0+', '') AS num_item_reserva_norm,
       regexp_replace(trim(cod_material), '^0+', '')     AS cod_material_norm,
       CASE WHEN trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0 THEN 'S' ELSE 'N' END           AS no_equivalente_mb25
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
ORDER BY 1, 2;

## 15. Checksum de linha

In [ ]:
-- 15. CHECKSUM
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), '')))) AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), '')))) AS linhas_100pct_identicas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014';

## 16. Amostra

> 🔒 Dados pessoais mascarados por hash.

In [ ]:
-- 16. AMOSTRA
SELECT `num_reserva`,
       `num_item_reserva`,
       `tp_registro`,
       `cod_material`,
       `desc_material`,
       `num_lote`,
       `cod_centro`,
       `cod_deposito`,
       `cod_centro_receptor`,
       `cod_deposito_receptor`,
       `dt_necessidade`,
       `dt_reserva`,
       `dt_criacao`,
       `qt_necessaria`,
       `qt_retirada`,
       `qt_saldo_reserva`,
       `sg_unidade_medida_basica`,
       `qt_um_registro`,
       `sg_unidade_medida_registro`,
       `qt_verificacao_disponibilidade`,
       `num_ordem`,
       `cod_operacao`,
       `tp_movimento`,
       `tp_necessidade`,
       `cod_debito_credito`,
       `num_ordem_referencia`,
       `num_ordem_servico`,
       `num_item_ordem_servico`,
       `st_reserva`,
       `ind_movimento_permitido`,
       `ind_item_eliminado`,
       `ind_registro_final`,
       `ind_item_dummy`,
       `ind_material_granel`,
       `cod_estoque_especial`,
       `cod_centro_custo`,
       `cod_elemento_pep`,
       `num_imobilizado`,
       `num_sub_imobilizado`,
       `num_diagrama_rede`,
       `num_ordem_cliente`,
       `num_item_ordem_cliente`,
       `num_divisao_ordem_cliente`,
       substring(md5(`nm_usuario`), 1, 8) AS `nm_usuario_hash`,
       substring(md5(`nm_recebedor_mercadoria`), 1, 8) AS `nm_recebedor_mercadoria_hash`,
       `desc_observacao`,
       `dateingest`,
       `yearingest`,
       `monthingest`
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
ORDER BY `num_reserva`, `num_item_reserva`
LIMIT 20;

## 17. Distribuição interna do centro 4014

In [ ]:
-- 17. DISTRIBUICAO POR tp_movimento
SELECT COALESCE(NULLIF(trim(`tp_movimento`), ''), '(vazio)') AS `tp_movimento`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(`tp_movimento`), ''), '(vazio)') ORDER BY linhas DESC LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR tp_necessidade
SELECT COALESCE(NULLIF(trim(`tp_necessidade`), ''), '(vazio)') AS `tp_necessidade`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(`tp_necessidade`), ''), '(vazio)') ORDER BY linhas DESC LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(`cod_deposito`), ''), '(vazio)') AS `cod_deposito`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(`cod_deposito`), ''), '(vazio)') ORDER BY linhas DESC LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR st_reserva
SELECT COALESCE(NULLIF(trim(`st_reserva`), ''), '(vazio)') AS `st_reserva`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(`st_reserva`), ''), '(vazio)') ORDER BY linhas DESC LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR ind_movimento_permitido
SELECT COALESCE(NULLIF(trim(`ind_movimento_permitido`), ''), '(vazio)') AS `ind_movimento_permitido`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(`ind_movimento_permitido`), ''), '(vazio)') ORDER BY linhas DESC LIMIT 40;

In [ ]:
-- 17.1 FRESHNESS
SELECT dateingest AS data_carga, COUNT(*) AS linhas, COUNT(DISTINCT num_reserva) AS reservas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' GROUP BY dateingest ORDER BY data_carga DESC;

## 18. Análises específicas — MB25

In [ ]:
-- 18.1 SALDO CALCULADO
SELECT COUNT(*) AS linhas,
       COUNT_IF(qt_saldo_reserva <> qt_necessaria - qt_retirada) AS formula_diverge,
       COUNT_IF(qt_saldo_reserva < 0) AS saldo_negativo,
       COUNT_IF(qt_retirada > qt_necessaria) AS retirada_a_maior,
       COUNT_IF(qt_retirada = 0) AS nada_retirado,
       COUNT_IF(qt_retirada > 0 AND qt_retirada < qt_necessaria) AS retirada_parcial
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014';

In [ ]:
-- 18.2 COMBINACOES SUSPEITAS
SELECT 'movimento NAO permitido MAS com retirada' AS situacao,
       COUNT_IF(trim(COALESCE(ind_movimento_permitido,'')) = '' AND qt_retirada > 0) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
UNION ALL SELECT 'movimento 261 SEM ordem de referencia',
       COUNT_IF(trim(tp_movimento) = '261' AND trim(COALESCE(num_ordem_referencia,'')) = '') FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
UNION ALL SELECT 'reserva com data APOS a necessidade',
       COUNT_IF(trim(dt_reserva) RLIKE '^[0-9]{8}$' AND dt_reserva <> '00000000' AND dt_reserva > dt_necessidade) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
UNION ALL SELECT 'mesma unidade MAS quantidade diferente',
       COUNT_IF(sg_unidade_medida_basica = sg_unidade_medida_registro AND qt_necessaria <> qt_um_registro) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
UNION ALL SELECT 'material sem descricao',
       COUNT_IF(desc_material IS NULL) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'
ORDER BY linhas DESC;

In [ ]:
-- 18.3 USUARIOS E RECEBEDORES (somente contagens - LGPD)
SELECT COUNT(DISTINCT nm_usuario) AS usuarios_distintos,
       COUNT_IF(trim(COALESCE(nm_usuario,'')) = '') AS sem_usuario,
       COUNT(DISTINCT nm_recebedor_mercadoria) AS recebedores_distintos,
       COUNT_IF(trim(COALESCE(nm_recebedor_mercadoria,'')) = '') AS sem_recebedor
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014';

## 19. EXTRAÇÕES — centro 4014

Use **Download → CSV** em cada resultado.

> 🔒 `nm_usuario` e `nm_recebedor_mercadoria` saem **mascarados por hash**. Para comparar esses dois
> campos com o SAP, use as contagens da seção 18.3.

### 19.1 Base completa do centro — comparar com RESB (SE16)

In [ ]:
-- 19.1 EXTRACAO BASE COMPLETA
SELECT `num_reserva`,
       `num_item_reserva`,
       `tp_registro`,
       `cod_material`,
       `desc_material`,
       `num_lote`,
       `cod_centro`,
       `cod_deposito`,
       `cod_centro_receptor`,
       `cod_deposito_receptor`,
       `dt_necessidade`,
       `dt_reserva`,
       `dt_criacao`,
       `qt_necessaria`,
       `qt_retirada`,
       `qt_saldo_reserva`,
       `sg_unidade_medida_basica`,
       `qt_um_registro`,
       `sg_unidade_medida_registro`,
       `qt_verificacao_disponibilidade`,
       `num_ordem`,
       `cod_operacao`,
       `tp_movimento`,
       `tp_necessidade`,
       `cod_debito_credito`,
       `num_ordem_referencia`,
       `num_ordem_servico`,
       `num_item_ordem_servico`,
       `st_reserva`,
       `ind_movimento_permitido`,
       `ind_item_eliminado`,
       `ind_registro_final`,
       `ind_item_dummy`,
       `ind_material_granel`,
       `cod_estoque_especial`,
       `cod_centro_custo`,
       `cod_elemento_pep`,
       `num_imobilizado`,
       `num_sub_imobilizado`,
       `num_diagrama_rede`,
       `num_ordem_cliente`,
       `num_item_ordem_cliente`,
       `num_divisao_ordem_cliente`,
       substring(md5(`nm_usuario`), 1, 8) AS `nm_usuario_hash`,
       substring(md5(`nm_recebedor_mercadoria`), 1, 8) AS `nm_recebedor_mercadoria_hash`,
       `desc_observacao`,
       `dateingest`,
       `yearingest`,
       `monthingest`
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
WHERE `cod_centro` = '4014'
ORDER BY `num_reserva`, `num_item_reserva`;

### 19.2 Equivalente MB25 — comparar com a tela MB25

Filtros: `XWAOK = X` e saldo em aberto. Na tela, use o intervalo de `dt_necessidade` da seção 3.2.

In [ ]:
-- 19.2 EXTRACAO EQUIVALENTE MB25
SELECT `num_reserva`,
       `num_item_reserva`,
       `tp_registro`,
       `cod_material`,
       `desc_material`,
       `num_lote`,
       `cod_centro`,
       `cod_deposito`,
       `cod_centro_receptor`,
       `cod_deposito_receptor`,
       `dt_necessidade`,
       `dt_reserva`,
       `dt_criacao`,
       `qt_necessaria`,
       `qt_retirada`,
       `qt_saldo_reserva`,
       `sg_unidade_medida_basica`,
       `qt_um_registro`,
       `sg_unidade_medida_registro`,
       `qt_verificacao_disponibilidade`,
       `num_ordem`,
       `cod_operacao`,
       `tp_movimento`,
       `tp_necessidade`,
       `cod_debito_credito`,
       `num_ordem_referencia`,
       `num_ordem_servico`,
       `num_item_ordem_servico`,
       `st_reserva`,
       `ind_movimento_permitido`,
       `ind_item_eliminado`,
       `ind_registro_final`,
       `ind_item_dummy`,
       `ind_material_granel`,
       `cod_estoque_especial`,
       `cod_centro_custo`,
       `cod_elemento_pep`,
       `num_imobilizado`,
       `num_sub_imobilizado`,
       `num_diagrama_rede`,
       `num_ordem_cliente`,
       `num_item_ordem_cliente`,
       `num_divisao_ordem_cliente`,
       substring(md5(`nm_usuario`), 1, 8) AS `nm_usuario_hash`,
       substring(md5(`nm_recebedor_mercadoria`), 1, 8) AS `nm_recebedor_mercadoria_hash`,
       `desc_observacao`,
       `dateingest`,
       `yearingest`,
       `monthingest`
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0
ORDER BY `num_reserva`, `num_item_reserva`;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'MB25' AS valor
UNION ALL SELECT 'CENARIO', 'ambiente', 'PRD'
UNION ALL SELECT 'CENARIO', 'tabela', 'vw_ds_log_mb25_reservas_material'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4014'
UNION ALL SELECT 'VOLUME', 'linhas base completa', format_number((SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 0)
UNION ALL SELECT 'VOLUME', 'linhas equivalente MB25', format_number((SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014' AND trim(COALESCE(ind_movimento_permitido,'')) = 'X' AND qt_saldo_reserva > 0), 0)
UNION ALL SELECT 'VOLUME', 'chaves reserva+item', format_number((SELECT COUNT(*) FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014')), 0)
UNION ALL SELECT 'QUALIDADE', 'linhas 100% identicas',
       format_number((SELECT COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), '')))) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 0)
UNION ALL SELECT 'QUALIDADE', 'linhas sem cabecalho', format_number((SELECT COUNT_IF(dt_reserva IS NULL) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material WHERE `cod_centro` = '4014'), 0)
ORDER BY bloco, item;

---

## Próximo passo

1. Rodar a **seção 3.1** e anotar o volume esperado na MB25.
2. Extrair no **SAP PRD** a MB25 do centro 4014 com: movimento permitido, somente em aberto e o
   intervalo de `dt_necessidade` da seção 3.2.
3. Baixar a **19.2** (e a 19.1 se for comparar com RESB). Anotar data/hora das duas extrações.
4. Enviar ao agente: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Comparar a 19.2 com a MB25 — **não** a 19.1
- [ ] Duplicatas puras e versões identificadas (seção 5) — no SAP cada item aparece uma vez
- [ ] Itens sem cabeçalho conferidos (seção 5.3)
- [ ] Zeros à esquerda normalizados (material 18, reserva 10, item 4, ordem 12)
- [ ] `00000000` tratado como data vazia (seção 12)
- [ ] Totais conferidos na unidade correta (seção 11)
- [ ] 🔒 Dados pessoais comparados só por contagem
